# Consumer Credit Risk Analysis — Data Preparation

## Objective

Prepare historical LendingClub consumer loan data for credit risk analysis and probability-of-default modeling.

This notebook covers:

- Loading the raw LendingClub dataset
- Selecting relevant credit risk variables
- Filtering the analysis period to 2014–2020
- Cleaning and transforming borrower and loan characteristics
- Defining the modeling population
- Creating the binary default target

### Default Definition

For modeling purposes:

- **Fully Paid = 0**
- **Charged Off = 1**

Loans without a resolved outcome are excluded from the modeling population.

In [2]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

## 1. Load LendingClub Data

The raw LendingClub dataset is large, so the file is read in 100,000-row chunks to reduce memory usage.

Only variables relevant to borrower characteristics, loan structure, and credit risk are retained. The analysis is limited to loans originated between 2014 and 2020.

In [3]:
columns = [
    "id",
    "issue_d",
    "loan_amnt",
    "term",
    "int_rate",
    "grade",
    "sub_grade",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "loan_status",
    "purpose",
    "dti",
    "delinq_2yrs",
    "fico_range_low",
    "fico_range_high",
    "inq_last_6mths",
    "open_acc",
    "revol_bal",
    "revol_util",
    "total_acc",
    "application_type"
]

chunks = []

for chunk in pd.read_csv(
    "../data/raw/Loan_status_2007-2020Q3.gzip",
    usecols=columns,
    chunksize=100000,
    low_memory=False
):
    chunk["issue_d"] = pd.to_datetime(
        chunk["issue_d"],
        format="%b-%Y",
        errors="coerce"
    )

    chunk = chunk[
        (chunk["issue_d"].dt.year >= 2014) &
        (chunk["issue_d"].dt.year <= 2020)
    ]

    chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)

print(df.shape)

(2694776, 23)


## 2. Define the Modeling Population

The probability-of-default analysis is limited to loans with resolved outcomes. Loans that are still current, late, or otherwise unresolved are excluded because their final repayment outcome is not yet known.

A binary target variable is created:

- **0 = Fully Paid**
- **1 = Charged Off**

In [4]:
model_df = df[
    df["loan_status"].isin(["Fully Paid", "Charged Off"])
].copy()

model_df["default_flag"] = (
    model_df["loan_status"] == "Charged Off"
).astype(int)

print("Modeling rows:", len(model_df))
print("\nDefault counts:")
print(model_df["default_flag"].value_counts())

print("\nDefault rate:")
print(model_df["default_flag"].mean())

Modeling rows: 1632364

Default counts:
default_flag
0    1305157
1     327207
Name: count, dtype: int64

Default rate:
0.2004497771330414


## 3. Feature Preparation

LendingClub provides the borrower's FICO score as a lower and upper range. A single FICO measure is calculated as the midpoint of these two values for use in risk analysis and modeling.

In [5]:
model_df["fico"] = (
    model_df["fico_range_low"] + model_df["fico_range_high"]
) / 2

model_df["fico"].describe()

count    1.632364e+06
mean     6.993565e+02
std      3.281795e+01
min      6.620000e+02
25%      6.720000e+02
50%      6.920000e+02
75%      7.170000e+02
max      8.475000e+02
Name: fico, dtype: float64

In [6]:
model_df["int_rate_num"] = (
    model_df["int_rate"]
    .str.replace("%", "", regex=False)
    .astype(float)
)

model_df["int_rate_num"].describe()

count    1.632364e+06
mean     1.307022e+01
std      4.876093e+00
min      5.310000e+00
25%      9.440000e+00
50%      1.262000e+01
75%      1.577000e+01
max      3.099000e+01
Name: int_rate_num, dtype: float64

## 4. Data Quality Check

Review missing values in the modeling dataset before exploratory analysis and model development.

In [7]:
model_df.isnull().sum().sort_values(ascending=False)

emp_length             108231
revol_util               1223
dti                      1108
inq_last_6mths              1
loan_amnt                   0
int_rate                    0
sub_grade                   0
term                        0
grade                       0
id                          0
verification_status         0
annual_inc                  0
home_ownership              0
loan_status                 0
purpose                     0
delinq_2yrs                 0
fico_range_low              0
issue_d                     0
fico_range_high             0
open_acc                    0
revol_bal                   0
total_acc                   0
application_type            0
default_flag                0
fico                        0
int_rate_num                0
dtype: int64

### Missing Data Assessment

Missing values are concentrated primarily in employment length, while the core variables used in the primary risk model have very limited or no missingness.

Rather than removing observations solely because of missing employment history, missing values will be handled as needed for specific analyses and modeling specifications. This preserves the maximum amount of usable portfolio data.

## 5. Export Modeling Dataset

Save the prepared modeling population for use in exploratory analysis and model development.

In [8]:
model_df.to_csv(
    "../data/processed/model_data.csv",
    index=False
)

print(model_df.shape)

(1632364, 26)
